In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier
import joblib


def metricas_classificacao(y_true, y_pred, target_names):
    report = classification_report(
        y_true,
        y_pred,
        target_names=target_names,
        output_dict=True,
        zero_division=0
    )
    tabela = pd.DataFrame(report).transpose()
    tabela = tabela.drop(index='accuracy', errors='ignore')
    tabela = tabela[['precision', 'recall', 'f1-score', 'support']]
    tabela['support'] = tabela['support'].astype(int)
    return tabela

# 1. Carga de Dados
df = pd.read_csv('../data/Dataset-Mental-Disorders.csv')
X = df.drop(['Patient Number', 'Expert Diagnose'], axis=1)
y = df['Expert Diagnose']

# 2. Encoding
le = LabelEncoder()
for col in X.columns:
    X[col] = le.fit_transform(X[col])
y = le.fit_transform(y)
target_names = le.classes_

print('Distribuicao das classes (sem reamostragem):')
print(pd.Series(le.inverse_transform(y)).value_counts().sort_index())

# 3. Divisao estratificada, preservando a distribuicao original das classes
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Hyperparameter tuning usando F1-macro como criterio
# Nao foi aplicada reamostragem sintetica porque as classes ja estao suficientemente balanceadas.
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0]
}

print('Iniciando busca exaustiva (GridSearch)...')
xgb = XGBClassifier(random_state=42, eval_metric='mlogloss')
grid_search = GridSearchCV(xgb, param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

# 5. Validacao por F1-macro no conjunto de treino original
cv_scores = cross_val_score(best_model, X_train, y_train, cv=10, scoring='f1_macro')
f1_macro_medio = cv_scores.mean()

# 6. Predicao e relatorio
y_pred = best_model.predict(X_test)

print('-' * 30)
print(f'F1-MACRO MEDIO (10-Fold CV): {f1_macro_medio:.2f}')
print(f'MELHORES PARAMETROS: {grid_search.best_params_}')
print('-' * 30)
print('\n=== RELATORIO DE CLASSIFICACAO ===')
print(metricas_classificacao(y_test, y_pred, target_names).to_string(float_format=lambda value: f'{value:.4f}'))

# 7. Grafico de Feature Importance
plt.figure(figsize=(10, 6))
importancias = pd.Series(best_model.feature_importances_, index=X.columns).sort_values()
importancias.plot(kind='barh', color='purple')
plt.title('Sintomas decisivos para o diagnostico')
plt.show()

# 8. Matriz de Confusao Final
plt.figure(figsize=(8, 6))
sns.heatmap(
    confusion_matrix(y_test, y_pred),
    annot=True,
    fmt='d',
    xticklabels=target_names,
    yticklabels=target_names,
    cmap='Purples'
)
plt.title('Matriz de Confusao Final')
plt.ylabel('Real')
plt.xlabel('Predito')
plt.show()

# 9. Salvando o progresso
joblib.dump(best_model, '../src/modelo_final_mental.pkl')
joblib.dump(le, '../src/label_encoder.pkl')
print('Modelo e encoder salvos em /src/.')
